# MyStyle Writer — LoRA Fine-Tuning (Colab T4)

Fine-tunes **Qwen2.5-1.5B-Instruct** with LoRA on the Wilde style corpus.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Smoke test (Day 3): set `SMOKE_TEST = True` — trains on a small slice, ~5 min.
Full run (Day 4): set `SMOKE_TEST = False` — full corpus, 2–3 epochs.

In [ ]:
SMOKE_TEST = True  # Day 3: True | Day 4: False

REPO_URL = "https://github.com/YOUR_USERNAME/mystyle-writer.git"  # <-- edit me

In [ ]:
!pip -q install transformers peft datasets accelerate bitsandbytes

import torch
assert torch.cuda.is_available(), "Enable the T4 GPU: Runtime -> Change runtime type"
print(torch.cuda.get_device_name(0))

## 1. Get the data
Clones your repo. If `data/processed/` is gitignored, upload the three JSONL files manually with the file browser instead (into `mystyle-writer/data/processed/`).

In [ ]:
import os
if not os.path.exists("mystyle-writer"):
    !git clone $REPO_URL
%cd mystyle-writer

import sys
sys.path.append("src")
import config

# If processed data isn't in the repo, regenerate it from raw (raw ships in the repo? if not, upload)
if not config.TRAIN_FILE.exists():
    print("Processed data missing — attempting to regenerate from data/raw/")
    !python src/prepare_data.py

In [ ]:
from datasets import load_dataset

data = load_dataset("json", data_files={
    "train": str(config.TRAIN_FILE),
    "validation": str(config.VAL_FILE),
})
if SMOKE_TEST:
    data["train"] = data["train"].select(range(min(64, len(data["train"]))))
    data["validation"] = data["validation"].select(range(min(16, len(data["validation"]))))
print(data)

## 2. Load base model in 4-bit and attach LoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(config.BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    config.BASE_MODEL, quantization_config=bnb, device_map="auto"
)
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=config.LORA_R,
    lora_alpha=config.LORA_ALPHA,
    lora_dropout=config.LORA_DROPOUT,
    target_modules=config.LORA_TARGET_MODULES,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 3. Tokenize

In [ ]:
def tokenize(batch):
    out = tokenizer(
        batch["text"], truncation=True, max_length=config.MAX_SEQ_LEN, padding=False
    )
    out["labels"] = [ids.copy() for ids in out["input_ids"]]
    return out

tokenized = data.map(tokenize, batched=True, remove_columns=["text"])

## 4. Train

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

run_name = "smoke-test" if SMOKE_TEST else "wilde-full"

args = TrainingArguments(
    output_dir=f"checkpoints/{run_name}",
    num_train_epochs=1 if SMOKE_TEST else config.EPOCHS,
    per_device_train_batch_size=config.BATCH_SIZE,
    gradient_accumulation_steps=config.GRAD_ACCUM,
    learning_rate=config.LEARNING_RATE,
    fp16=True,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
trainer.train()

## 5. Save the adapter

In [ ]:
adapter_dir = f"models/adapters/{run_name}"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("Saved to", adapter_dir)

# Zip it for download (Colab storage is temporary!)
!zip -rq {run_name}-adapter.zip {adapter_dir}
from google.colab import files
files.download(f"{run_name}-adapter.zip")

## 6. Quick before/after sanity check

In [ ]:
prompt = "Write a short passage about a stranger arriving in a quiet town."

def generate(m):
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        return_tensors="pt", add_generation_prompt=True
    ).to(m.device)
    with torch.no_grad():
        out = m.generate(inputs, max_new_tokens=config.GEN_MAX_NEW_TOKENS,
                         temperature=config.GEN_TEMPERATURE, top_p=config.GEN_TOP_P,
                         do_sample=True, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

print("=== FINE-TUNED (adapter active) ===")
print(generate(model))

with model.disable_adapter():
    print("\n=== BASE (adapter disabled) ===")
    print(generate(model))

## 7. Batch generation + evaluation (run after the FULL training, Day 4/6 work)

Generates paired base/tuned outputs for 10 neutral prompts, then scores both
against the held-out Wilde split. Produces the files the Streamlit app's
demo mode needs (`evaluation/generated_*.jsonl`, `evaluation/results.json`).

In [ ]:
!pip -q install sentence-transformers

!python src/inference.py --adapter models/adapters/{run_name} \
    --out-base evaluation/generated_base.jsonl \
    --out-tuned evaluation/generated_tuned.jsonl \
    --out-prompted evaluation/generated_prompted.jsonl \
    --samples-per-prompt 2

In [ ]:
!python evaluation/style_eval.py \
    --base evaluation/generated_base.jsonl \
    --tuned evaluation/generated_tuned.jsonl \
    --prompted evaluation/generated_prompted.jsonl \
    --out evaluation/results.json

In [ ]:
# Download everything the app + README need
!zip -q eval-outputs.zip evaluation/generated_*.jsonl evaluation/results.json
from google.colab import files
files.download("eval-outputs.zip")